In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

STORAGE_ACCOUNT = "vjretailflow01"
STORAGE_KEY = dbutils.secrets.get(scope="retailflow", key="adls-key")

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

BRONZE_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net"
SILVER_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net"

print("Setup complete")

In [0]:
order_items_bronze = spark.read.format("delta") \
    .load(f"{BRONZE_PATH}/order_items")

order_items_bronze.printSchema()
display(order_items_bronze.limit(3))

In [0]:
def transform_order_items(df):
    return df \
        .filter(F.col("order_id").isNotNull()) \
        .filter(F.col("product_id").isNotNull()) \
        .filter(F.col("seller_id").isNotNull()) \
        .filter(F.col("price") > 0) \
        .withColumn("order_id", F.col("order_id").cast(T.StringType())) \
        .withColumn("order_item_id", F.col("order_item_id").cast(T.IntegerType())) \
        .withColumn("product_id", F.col("product_id").cast(T.StringType())) \
        .withColumn("seller_id", F.col("seller_id").cast(T.StringType())) \
        .withColumn("shipping_limit_date", F.col("shipping_limit_date").cast(T.TimestampType())) \
        .withColumn("price", F.round(F.col("price"),2)) \
        .withColumn("freight_value", F.round(F.col("freight_value"),2)) \
        .withColumn("total_value", F.round(F.col("price") + F.col("freight_value"),2)) \
        .withColumn("_order_items_silver_processed_at", F.current_timestamp()) \
        .drop("_ingested_at", "_source_file", "_ingestion_date")
order_items_silver = transform_order_items(order_items_bronze)
display(order_items_silver.limit(3))

In [0]:
print(f"Total rows: {order_items_silver.count():,}")
print(f"Null order ids: {order_items_silver.filter(F.col("order_id").isNull())}")
print(f"Zero price rows: {order_items_silver.filter(F.col("price") <= 0).count()}")

print("\nRevenue Summary:")
display(order_items_silver.agg(
    F.round(F.sum("price"),2).alias("total_revenue"),
    F.round(F.avg("price"),2).alias("avg_order_price"),
    F.round(F.max("price"),2).alias("max_price"),
    F.round(F.min("price"),2).alias("min_price")
))

order_items_silver.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("seller_id") \
    .save(f"{SILVER_PATH}/order_items/")
print("Silver order items written successfully!")